# Applying DeepRV to Gaussian Processes for IPMs

In this notebook, we investigate how the computational cost of training **DeepRV** scales with the size of data generated from a Gaussian process. We consider simulated datasets of size $n = 250, 500, 750,$ and $1,\!000$.

The data are simulated with the vital rates modelled as Gaussian processes (GPs) rather than generalized linear models (GLMs).

Let $s \in \mathbb{R}^n$ denote a fixed set of input locations (the sizes of all individuals in the dataset), and let $k_\theta(\cdot,\cdot)$ be a covariance kernel with parameters $\theta$. We assume

$$
f(s) \sim \mathcal{GP}(0, K_\theta),
\qquad
(K_\theta)_{ij} = k_\theta(s_i,s_j).
$$

Equivalently, a sample from the Gaussian process can be written as

$$
f(s) = L_\theta z,
\qquad
z \sim \mathcal{N}(0, I_n),
\qquad
L_\theta = \operatorname{chol}(K_\theta),
$$

where $L_\theta$ is the Cholesky factor of the covariance matrix.

Computing the Cholesky decomposition of an $n \times n$ covariance matrix requires $O(n^3)$ operations, making it the computational bottleneck for many GP-based methods. DeepRV seeks to learn a neural network approximation of this mapping,

$$
\mathrm{DeepRV}(\theta, z) \approx f(s),
$$

allowing GP samples to be generated without explicitly performing a Cholesky decomposition.

We use the gMLP formulation of DeepRV adapted from `dl4bi/benchmarks/vae/deep_rv_example.py`. Since the input locations are fixed, the network takes the full vector of input locations $s$, together with the kernel parameters $\theta$ and latent Gaussian vector $z$, as input.

The objective of this notebook is to investigate how the computational cost of training DeepRV scales with the size of the Gaussian process. Separate DeepRV models are trained for each dataset size, and their training times are compared to assess the scalability of the approach for the larger Gaussian processes encountered in integral projection models (IPMs).

In [1]:
import sys

sys.path.append("benchmarks/vae")

from pathlib import Path
from typing import Callable, Optional, Union
import jax
import time
import jax.numpy as jnp
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import optax
from jax import Array, jit, random
from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS, Predictive, init_to_median
from dl4bi_sps.kernels import rbf
from dl4bi_sps.utils import build_grid
from utils.plot_utils import plot_infer_trace

import wandb
from dl4bi.core.model_output import VAEOutput
from dl4bi.core.train import cosine_annealing_lr, train
from dl4bi.vae import gMLPDeepRV
from dl4bi.vae.train_utils import deep_rv_train_step, generate_surrogate_decoder

from deeprv_utils import hmc, gen_train_dataloader, inference_model, valid_step, train_deeprv, run_gp_inference, run_deeprv_inference, compute_metrics

/opt/anaconda3/envs/dl4bi-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/dl4bi-env/lib/python3.11/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [ ]:
priors = {
    "ls": dist.HalfNormal(5.0), 
    "r":dist.HalfNormal(5.0)
    }

wandb.init(mode="disabled")

NameError: name 'dist' is not defined

In [3]:
dataset_sizes = [250]  #,500,750,1000]
datasets = {}

for n in dataset_sizes:
    datasets[n] = pd.read_csv(f"gp_sample{n}.csv")
    print(datasets[n].head())
    print(f"\nShape: {datasets[n].shape}\n")
    print(datasets[n].describe())

       id  year         z        z1  age  survived  reproduced  n_seeds
0  445141    39  1.068144  1.716993    2       1.0           0      NaN
1  110162    28  0.071170       NaN    0       0.0           0      NaN
2  266958    34  1.331782  2.654812    1       1.0           0      NaN
3  327160    35 -1.156128       NaN    0       0.0           0      NaN
4  367965    37  1.078800       NaN    1       0.0           0      NaN

Shape: (250, 8)

                  id        year           z          z1         age  \
count     250.000000  250.000000  250.000000  111.000000  250.000000   
mean   352209.496000   34.220000    0.513948    1.553747    0.692000   
std    201441.761066    5.854631    1.112116    0.920278    1.139443   
min      5514.000000    8.000000   -1.843070   -0.610952    0.000000   
25%    174347.750000   31.000000   -0.281761    1.008739    0.000000   
50%    342747.000000   36.000000    0.418685    1.590690    0.000000   
75%    522540.000000   39.000000    1.227178  

In [ ]:
def generate_data(dataset):
    s = dataset["z"]
    s_jnp = jnp.asarray(s.to_numpy(dtype="float32"))
    s = s_jnp[:, None]
    return s

## Testing 

## Saving Trained DeepRV Models

The scaling experiment trains a new DeepRV surrogate for each dataset size. By default, the trained neural network only exists in memory during the notebook run. Once the function finishes, the trained parameters are lost unless we explicitly save them.

The code below saves the trained DeepRV `TrainState` after training. This state contains the learned neural-network parameters, optimizer state, and any additional model variables needed to reconstruct the surrogate decoder later.

For each dataset size, the checkpoint is saved under:

```text
results/deeprv_scaling/N_<dataset_size>/model.ckpt

In [8]:
import json
from pathlib import Path
from omegaconf import DictConfig
from dl4bi.core.train import save_ckpt

In [ ]:
def save_deeprv_training_output(
    state,
    save_dir,
    *,
    s,
    train_time,
    num_train_steps,
    num_blks=2,
    conditional_names=("ls"),
    kernel_name="rbf",
):
    """
    Save the trained DeepRV state, model configuration, locations, and metadata.
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path = save_dir / "model.ckpt"
    locations_path = save_dir / "s.npy"
    metadata_path = save_dir / "metadata.json"

    # This tells load_ckpt() how to reconstruct the neural-network architecture.
    model_config = DictConfig({
        "model": {
            "_target_": "dl4bi.vae.gMLPDeepRV",
            "num_blks": int(num_blks),
        }
    })

    save_ckpt(
        state,
        model_config,
        ckpt_path,
    )

    # Locations are inputs to DeepRV, so they are not part of the TrainState.
    s_np = np.asarray(jax.device_get(s))
    np.save(locations_path, s_np)

    metadata = {
        "model_name": "gMLPDeepRV",
        "num_blks": int(num_blks),
        "dataset_size": int(s_np.shape[0]),
        "input_dimension": int(s_np.shape[1]),
        "s_shape": list(s_np.shape),
        "conditional_names": list(conditional_names),
        "kernel_name": kernel_name,
        "train_time_seconds": float(train_time),
        "num_train_steps": int(num_train_steps),
        "checkpoint_path": str(ckpt_path),
        "locations_path": str(locations_path),
    }

    with metadata_path.open("w") as f:
        json.dump(metadata, f, indent=2)

    return ckpt_path

In [11]:
def evaluate_deeprv_surrogate(
    rng,
    s,
    priors,
    surrogate_decoder,
    *,
    batch_size=128,
    num_eval_batches=20,
):
    """
    Evaluate a trained DeepRV surrogate against exact GP draws.

    The function creates fresh validation data from the GP prior, runs the
    trained DeepRV surrogate on the same latent inputs, and compares the exact
    GP draws ``f_true`` with the neural approximation ``f_hat``.

    Parameters
    ----------
    rng : jax.Array
        Random key used to generate validation batches.

    s : jax.Array
        Input locations used by DeepRV. Shape should be ``(N, input_dim)``.

    priors : dict
        Dictionary of kernel hyperparameter priors. Expected keys are ``"ls"``
        for length scale and ``"r"`` for variance/amplitude.

    surrogate_decoder : callable
        Trained DeepRV decoder returned by ``train_deeprv``.

    batch_size : int
        Number of GP draws per validation batch.

    num_eval_batches : int
        Number of validation batches to evaluate. More batches give a more
        stable estimate, but take longer.

    Returns
    -------
    dict
        Dictionary containing:
        - ``relative_rmse``: one overall relative RMSE value;
        - ``f_true``: exact GP draws;
        - ``f_hat``: DeepRV approximations;
        - ``s``: input locations;
        - ``ls``: length scale used for each validation draw;
        - ``relative_rmse_by_draw``: relative RMSE for each individual draw.
    """
    loader = gen_train_dataloader(
        s,
        priors,
        batch_size=batch_size,
    )

    batches = loader(rng)

    f_true_all = []
    f_hat_all = []
    ls_all = []

    for _ in range(num_eval_batches):
        batch = next(batches)

        f_true = batch["f"]

        f_hat = surrogate_decoder(
            batch["z"],
            batch["conditionals"],
            s=batch["s"],
        )

        # DeepRV may return shape (batch, N, 1). Squeeze gives (batch, N).
        f_hat = jnp.squeeze(f_hat)

        # In the current dataloader, one length scale is used for the whole
        # batch, so we repeat it once per draw.
        ls = batch["conditionals"][0]
        ls_repeated = jnp.repeat(ls, f_true.shape[0])

        f_true_all.append(f_true)
        f_hat_all.append(f_hat)
        ls_all.append(ls_repeated)

    f_true_all = jnp.concatenate(f_true_all, axis=0)
    f_hat_all = jnp.concatenate(f_hat_all, axis=0)
    ls_all = jnp.concatenate(ls_all, axis=0)

    residuals = f_hat_all - f_true_all

    rmse = jnp.sqrt(jnp.mean(residuals ** 2))
    sd_true = jnp.std(f_true_all)

    # Relative RMSE is unitless. For example, 0.05 means the network error is
    # about 5% of the natural standard deviation of the true GP draws.
    relative_rmse = rmse / sd_true

    # Compute a relative RMSE for each GP draw. This is useful for plotting
    # accuracy against length scale.
    rmse_by_draw = jnp.sqrt(jnp.mean(residuals ** 2, axis=1))
    sd_by_draw = jnp.std(f_true_all, axis=1)
    relative_rmse_by_draw = rmse_by_draw / sd_by_draw

    return {
        "relative_rmse": float(relative_rmse),
        "f_true": np.asarray(jax.device_get(f_true_all)),
        "f_hat": np.asarray(jax.device_get(f_hat_all)),
        "s": np.asarray(jax.device_get(s)).squeeze(),
        "ls": np.asarray(jax.device_get(ls_all)),
        "relative_rmse_by_draw": np.asarray(
            jax.device_get(relative_rmse_by_draw)
        ),
    }

In [ ]:
def plot_deeprv_surrogate_diagnostics(
    eval_output,
    *,
    num_examples=6,
    num_lengthscale_bins=8,
):
    """
    Plot diagnostic figures for a trained DeepRV surrogate.

    The function creates three plots:

    1. ``f_true`` versus ``f_hat`` scatter plot.
       This checks whether predictions lie close to the diagonal line.

    2. Example GP function draws.
       This compares exact GP sample paths against DeepRV sample paths.

    3. Relative RMSE versus length scale.
       This checks whether DeepRV performs worse for particular length-scale
       regimes.

    Parameters
    ----------
    eval_output : dict
        Output from ``evaluate_deeprv_surrogate``.

    num_examples : int
        Number of individual GP draws to show in the sample-path plot.

    num_lengthscale_bins : int
        Number of bins used for the length-scale diagnostic plot.
    """
    f_true = eval_output["f_true"]
    f_hat = eval_output["f_hat"]
    s = eval_output["s"]
    ls = eval_output["ls"]
    relative_rmse_by_draw = eval_output["relative_rmse_by_draw"]

    # Sort locations so the function-draw plots are shown as smooth curves.
    order = np.argsort(s)
    s_sorted = s[order]

    # ------------------------------------------------------------
    # Plot 1: f_true versus f_hat scatter.
    # ------------------------------------------------------------
    plt.figure(figsize=(6, 6))

    plt.scatter(
        f_true.ravel(),
        f_hat.ravel(),
        alpha=0.15,
        s=8,
    )

    lim_min = min(f_true.min(), f_hat.min())
    lim_max = max(f_true.max(), f_hat.max())

    plt.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        color="black",
        linewidth=1,
        label="Perfect Prediction",
    )

    plt.xlabel("True GP draw")
    plt.ylabel("DeepRV Approximation")
    plt.title("DeepRV Approximation Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # Plot 2: example function draws.
    # ------------------------------------------------------------
    num_examples = min(num_examples, f_true.shape[0])

    fig, axes = plt.subplots(
        num_examples,
        1,
        figsize=(8, 2.2 * num_examples),
        sharex=True,
    )

    if num_examples == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        ax.plot(
            s_sorted,
            f_true[i, order],
            label="True GP",
            linewidth=2,
        )

        ax.plot(
            s_sorted,
            f_hat[i, order],
            label="DeepRV",
            linewidth=2,
            linestyle="--",
        )

        ax.set_ylabel(f"Draw {i + 1}")

        if i == 0:
            ax.legend()

    axes[-1].set_xlabel("s")
    fig.suptitle("Example GP Draws: True vs DeepRV")
    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # Plot 3: relative RMSE versus length scale.
    # ------------------------------------------------------------
    bins = np.linspace(ls.min(), ls.max(), num_lengthscale_bins + 1)
    bin_ids = np.digitize(ls, bins) - 1

    bin_centres = []
    bin_means = []

    for bin_id in range(num_lengthscale_bins):
        mask = bin_ids == bin_id

        if np.any(mask):
            bin_centres.append(0.5 * (bins[bin_id] + bins[bin_id + 1]))
            bin_means.append(relative_rmse_by_draw[mask].mean())

    plt.figure(figsize=(7, 5))

    plt.plot(
        bin_centres,
        bin_means,
        "-o",
    )

    plt.xlabel("Length Scale")
    plt.ylabel("Relative RMSE")
    plt.title("DeepRV Error by Length Scale")
    plt.tight_layout()
    plt.show()

In [ ]:
def train_and_evaluate_deeprv(
    dataset,
    priors,
    seed=0,
    num_train_steps=100_000,
    save_root="results/train_deeprv",
):
    """
    Run one DeepRV-vs-GP scaling experiment and save the trained DeepRV state.

    This function trains a DeepRV surrogate for the given dataset, saves the
    trained neural-network state to disk, then runs GP and DeepRV inference.

    Notes
    -----
    Expected data shapes after ``generate_survival_data``:

    - ``s.shape == (N, spatial_dim)``
    - ``y.shape == (N,)``

    Returns
    -------
    dict
        Metrics from the scaling experiment, including the checkpoint path.
    """
    s = generate_data(dataset)

    rng = random.key(seed)
    rng_train, rng_eval = random.split(rng, 2)

    surrogate_decoder, train_time, state = train_deeprv(
        rng_train,
        s,
        priors,
        num_steps=num_train_steps,
    )

    eval_output = evaluate_deeprv_surrogate(
        rng_eval,
        s,
        priors,
        surrogate_decoder,
        batch_size=128,
        num_eval_batches=20,
    )

    print(
        "DeepRV validation relative RMSE:",
        eval_output["relative_rmse"],
    )

    plot_deeprv_surrogate_diagnostics(
        eval_output,
        num_examples=6,
        num_lengthscale_bins=8,
    )

    checkpoint_dir = Path(save_root) / f"N_{len(s)}"
    checkpoint_path = save_deeprv_training_output(
    state,
    checkpoint_dir,
    s=s,
    train_time=train_time,
    num_train_steps=num_train_steps,
    num_blks=2,
    conditional_names=("ls",),
    kernel_name="rbf",
    )

    return surrogate_decoder, state

In [ ]:
trained_networks = {}

for n in dataset_sizes:
    print(f"Running N = {n}")

    run = wandb.init(
        project="deeprv-training",
        name=f"deeprv-N-{n}",
        config={
            "requested_dataset_size": n,
            "num_train_steps": 100_000,
            "batch_size": 32,
            "learning_rate": 1e-3,
            "weight_decay": 1e-2,
            "num_blks": 2,
            "kernel": "rbf",
            "variance": 1.0,
            "lengthscale_prior": "HalfNormal(5.0)",
        },
    )

    try:
        trained = train_and_evaluate_deeprv(
            dataset=datasets[n],
            priors=priors,
            seed=0,
            num_train_steps=100_000,
        )

        trained_networks[f"NN_{n}"] = trained

    finally:
        run.finish()